In [ ]:
!pip install praw pandas

In [ ]:
from google.colab import userdata
CLIENT_ID = userdata.get("reddit_client_id")
CLIENT_SECRET = userdata.get("reddit_client_secret")
USER_AGENT = userdata.get("reddit_user_agent")

In [ ]:
import praw

reddit = praw.Reddit(
    client_id     = CLIENT_ID,
    client_secret = CLIENT_SECRET,
    user_agent    = USER_AGENT
)

In [ ]:
import pandas as pd

# ---------------------------
# CONFIG
# ---------------------------
brands = ["Swiggy", "Zomato"]          # Brands to fetch
comments_per_brand = 2000              # Approx number of comments per brand
subreddits_to_search = "india+food+DeliveryServices"  # Optional, limit search to relevant subreddits

all_comments = []

# ---------------------------
# FETCH COMMENTS
# ---------------------------
for brand in brands:
    count = 0
    for submission in reddit.subreddit(subreddits_to_search).search(brand, sort='new', limit=None):
        submission.comments.replace_more(limit=0)
        for comment in submission.comments.list():
            if count >= comments_per_brand:
                break
            # Include only comments that mention the brand
            if brand.lower() in comment.body.lower():
                all_comments.append({
                    "text": comment.body,
                    "brand": brand,
                    "comment_id": comment.id,
                    "score": comment.score,
                    "created_utc": comment.created_utc,
                    "author": str(comment.author),
                    "post_title": submission.title  # reference only
                })
                count += 1
        if count >= comments_per_brand:
            break

# ---------------------------
# CLEAN & SAVE
# ---------------------------
df = pd.DataFrame(all_comments)

# Remove exact duplicates
df.drop_duplicates(subset=['text', 'brand'], inplace=True)

# Shuffle data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to CSV
csv_filename = "candy.csv"
df.to_csv(csv_filename, index=False)
print(f"Fetched {len(df)} comments for Swiggy and Zomato combined.")

# ---------------------------
# DOWNLOAD CSV (Colab)
# ---------------------------
from google.colab import files
files.download(csv_filename)


In [ ]:
import pandas as pd

# --- Load all files ---
df1 = pd.read_csv("reddit_comments_sw_zom.csv")
df2 = pd.read_csv("candy.csv")
df3 = pd.read_csv("synthetic_reviews.csv")        # assuming .csv
df4 = pd.read_csv("synthetic_reviews_v2.csv")

# --- Concatenate / Merge (stack rows) ---
final_df = pd.concat([df1, df2, df3, df4], ignore_index=True)

# --- Save the merged file ---
final_df.to_csv("merged_reviews.csv", index=False)

print("Merging completed! Final shape:", final_df.shape)


In [ ]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download VADER lexicon (run once)
nltk.download('vader_lexicon')

# Load merged file
df = pd.read_csv("merged_reviews.csv")   # <-- updated filename

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Auto sentiment scoring
df['compound'] = df['text'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

# Auto-label sentiment
df['label'] = 0
df.loc[df['compound'] > 0.2, 'label'] = 1     # Positive
df.loc[df['compound'] < -0.2, 'label'] = -1    # Negative

# Show sample
print(df[['text', 'brand', 'compound', 'label']].head(10))

# Save labeled dataset
df.to_csv("merged_reviews_labeled.csv", index=False)  # <-- updated output filename

print("\n✅ Auto-labelling complete! File saved as 'merged_reviews_labeled.csv'")


In [ ]:
import pandas as pd

df = pd.read_csv("merged_reviews_labeled.csv")

# Count each class
print(df['label'].value_counts())

# Percentage distribution
print("\nPercentage distribution:")
print(df['label'].value_counts(normalize=True) * 100)


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# ---------------------------------------
# Load your labeled data
# ---------------------------------------
df = pd.read_csv("merged_reviews_labeled.csv")

# ---------------------------------------
# 1. Lowercase
# ---------------------------------------
def to_lower(text):
    return text.lower()

# ---------------------------------------
# 2. Remove URLs
# ---------------------------------------
def remove_urls(text):
    return re.sub(r'http\S+|www.\S+', '', text)

# ---------------------------------------
# 3. Remove emojis
# ---------------------------------------
emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport
                           u"\U0001F1E0-\U0001F1FF"  # flags
                           "]+", flags=re.UNICODE)

def remove_emojis(text):
    return emoji_pattern.sub(r'', text)

# ---------------------------------------
# 4. Remove punctuation and numbers
# ---------------------------------------
def remove_punc_num(text):
    return re.sub(r'[^a-zA-Z\s]', ' ', text)

# ---------------------------------------
# 5. Remove stopwords
# ---------------------------------------
def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in stop_words])

# ---------------------------------------
# 6. Clean extra spaces
# ---------------------------------------
def clean_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

# ---------------------------------------
# Combine all steps
# ---------------------------------------
def preprocess(text):
    text = str(text)
    text = to_lower(text)
    text = remove_urls(text)
    text = remove_emojis(text)
    text = remove_punc_num(text)
    text = remove_stopwords(text)
    text = clean_spaces(text)
    return text

# ---------------------------------------
# Apply preprocessing
# ---------------------------------------
df['clean_text'] = df['text'].apply(preprocess)

# ---------------------------------------
# Keep only clean text + label
# ---------------------------------------
final_df = df[['clean_text', 'label']]

# Save file
final_df.to_csv("clean_preprocessed_data.csv", index=False)

print("\n✅ Preprocessing complete! Saved as 'clean_preprocessed_data.csv'")
print(final_df.head())


In [ ]:
# =======================================
# 3-CLASS SENTIMENT MODEL (LSTM + GLOVE + CLASS WEIGHTS)
# =======================================

# ---------------------------
# 0️⃣ IMPORTS
# ---------------------------
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam

# ---------------------------
# 1️⃣ LOAD DATA
# ---------------------------
df = pd.read_csv("clean_preprocessed_data.csv")   # <-- cleaned dataset

X = df['clean_text'].astype(str)
y = df['label']         # -1, 0, 1

# Convert labels to 0,1,2 (required for softmax)
label_map = {-1: 0, 0: 1, 1: 2}
y = y.map(label_map)

# ---------------------------
# 2️⃣ TRAIN-TEST SPLIT
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------
# 3️⃣ TOKENIZATION & PADDING
# ---------------------------
vocab_size = 15000
maxlen = 120

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen, padding='post')

print("Tokenization complete.")

# ---------------------------
# 4️⃣ LOAD GLOVE EMBEDDINGS
# ---------------------------
if not os.path.exists("glove.6B.100d.txt"):
    print("Downloading GloVe...")
    !wget http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip

embeddings_index = {}
with open("glove.6B.100d.txt", encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        vector = embeddings_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector

print("GloVe Loaded.")

# ---------------------------
# 5️⃣ CLASS WEIGHTS
# ---------------------------
classes = np.array([0, 1, 2])   # mapped labels
weights = compute_class_weight('balanced', classes=classes, y=y_train)

class_weights = {int(cls): float(w) for cls, w in zip(classes, weights)}
print("Class Weights:", class_weights)

# ---------------------------
# 6️⃣ BUILD MODEL (3-CLASS)
# ---------------------------
inputs = Input(shape=(maxlen,))

x = Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    input_length=maxlen,
    trainable=True
)(inputs)

x = Bidirectional(LSTM(128, return_sequences=True,
                       dropout=0.3, recurrent_dropout=0.3))(x)
x = BatchNormalization()(x)

x = Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3))(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.4)(x)

outputs = Dense(3, activation='softmax')(x)  # 3 classes

model = Model(inputs, outputs)

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(1e-3),
    metrics=['accuracy']
)

model.summary()

# ---------------------------
# 7️⃣ TRAIN MODEL
# ---------------------------
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint("tri_sentiment_best_model.keras", save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_test_pad, y_test),
    epochs=15,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks
)

# ---------------------------
# 8️⃣ SAVE MODEL & TOKENIZER
# ---------------------------
model.save("tri_sentiment_final_model.keras")

with open("tri_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("\n✅ MODEL & TOKENIZER SAVED SUCCESSFULLY!")

# ---------------------------
# 9️⃣ PREDICT FUNCTION (3-CLASS)
# ---------------------------
reverse_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=maxlen)
    pred = model.predict(pad)[0]
    label = np.argmax(pred)

    print(f"Text: {text}")
    print(f"Sentiment: {reverse_map[label]} (score={pred[label]:.3f})")

# Example
predict_sentiment("Food was okay, delivery was late but not too bad.")


In [ ]:
def predict_sentiment(text):
    # Convert text → sequence
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=maxlen, padding='post', truncating='post')

    # Model prediction
    pred = model.predict(padded)[0]

    # Convert to class label
    label = pred.argmax()  # 0 = Negative, 1 = Neutral, 2 = Positive

    # Map numbers → words
    mapping = {
        0: "Negative 😞",
        1: "Neutral 😐",
        2: "Positive 😀"
    }

    confidence = pred[label]

    print("\nInput Text:", text)
    print("Predicted Class:", mapping[label])
    print(f"Confidence Score: {confidence:.4f}")

    return mapping[label]

In [ ]:
predict_sentiment("The food was delicious and the delivery was fast.")
predict_sentiment("It was okay, nothing special but not bad either.")
predict_sentiment("Terrible experience, the order never arrived.")
predict_sentiment("I loved the packaging, very neat and clean.")
predict_sentiment("The delivery guy was rude and the food was cold.")
predict_sentiment("Service was fine, but I expected it to be faster.")
predict_sentiment("Amazing taste! I will definitely order again.")
predict_sentiment("Not impressed, but it's acceptable for the price.")
predict_sentiment("Worst experience ever, completely disappointed.")
predict_sentiment("Everything was perfect, I have no complaints at all.")


In [ ]:
# These are the values, I got as a result while training the model. Actually my runtime got disconnected. So, I simply just copied the values here.


import matplotlib.pyplot as plt

# Training log values extracted from user message
epochs = list(range(1, 10))

train_acc = [0.4707, 0.7263, 0.7920, 0.8316, 0.8437, 0.8777, 0.8943, 0.9098, 0.9241]
val_acc   = [0.7549, 0.8188, 0.8410, 0.8254, 0.8500, 0.8767, 0.8747, 0.8893, 0.8732]

train_loss = [1.0218, 0.6268, 0.4833, 0.4153, 0.3724, 0.3219, 0.2834, 0.2423, 0.1955]
val_loss   = [0.5869, 0.4563, 0.4034, 0.4049, 0.3799, 0.3363, 0.3575, 0.3760, 0.3965]

# 1. Accuracy plot
plt.figure(figsize=(8,5))
plt.plot(epochs, train_acc, label="Train Accuracy")
plt.plot(epochs, val_acc, label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 2. Loss plot
plt.figure(figsize=(8,5))
plt.plot(epochs, train_loss, label="Train Loss")
plt.plot(epochs, val_loss, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()
